In [0]:
from pyspark.sql.functions import year, month, sum, col, to_date, coalesce, lit

df = spark.table("edl_dev.databricks_ci_cd_src.account_statement_2026")

result = (
    df.filter(col("_c0") != "Date")
      .withColumn("date_col", to_date(col("_c0"), "dd/MM/yyyy"))
      .withColumn("_c3_double", col("_c3").cast("double"))
      .withColumn("_c4_double", col("_c4").cast("double"))
      .groupBy(year(col("date_col")).alias("year"), month(col("date_col")).alias("month"))
      .agg(
          sum(col("_c3_double")).alias("debit"),
          sum(col("_c4_double")).alias("credit"),
          sum((coalesce(col("_c4_double"), lit(0)) - coalesce(col("_c3_double"), lit(0))).cast("int")).alias("diff_int")
      )
)

# display(result)

In [0]:
result.write.format("delta").mode("overwrite").option("path", "s3://s3-databrick-final/core/").saveAsTable(f"{catalog}.{schema_core}.account_statement_2026")